In [ ]:
import sys, os
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv("../.env")

from arxiv_agent.observability import check_connection, get_client, flush

assert check_connection(), "Langfuse not reachable at localhost:3000 — is it running?"
print("Langfuse connection OK")
print(f"Langfuse UI: {os.environ['LANGFUSE_BASE_URL']}")

In [ ]:
from arxiv_agent.observability import start_trace, flush
from arxiv_agent.agent import run_agent

question = "What is the main contribution of the Attention Is All You Need paper?"

trace = start_trace(question)
result = run_agent(question, trace=trace)
flush()

print(f"\nAnswer:\n{result['answer']}")
print(f"\nLangfuse trace: {os.environ['LANGFUSE_BASE_URL']}/trace/{result['trace_id']}")

In [ ]:
from langfuse import Langfuse
import time

lf = get_client()
time.sleep(1)

raw = lf.get_trace(result["trace_id"])
print(f"Trace ID: {raw.id}")
print(f"Input: {raw.input}")
print(f"Output: {raw.output}")
print(f"\nSpans ({len(raw.observations)}):")
for obs in raw.observations:
    print(f"  [{obs.type}] {obs.name} — {obs.metadata}")

In [ ]:
import json
from pathlib import Path
from arxiv_agent.agent import build_graph, SYSTEM_PROMPT
from langchain_core.messages import HumanMessage, SystemMessage

cases = json.loads(Path("../data/eval_cases.json").read_text())
results = []

for case in cases:
    print(f"Running {case['id']}: {case['question'][:60]}...")
    trace = start_trace(case["question"])

    graph = build_graph()
    msgs = graph.invoke({
        "messages": [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=case["question"])],
        "trace": trace,
    })

    final_answer = msgs["messages"][-1].content
    trace.update(output=final_answer)

    tool_calls_made = []
    for m in msgs["messages"]:
        if hasattr(m, "tool_calls") and m.tool_calls:
            for tc in m.tool_calls:
                tool_calls_made.append({"name": tc["name"], "args": tc["args"]})

    results.append({
        "id": case["id"],
        "category": case["category"],
        "question": case["question"],
        "answer": final_answer,
        "trace_id": trace.id,
        "tool_calls_made": tool_calls_made,
    })
    flush()

Path("../data/run_results.json").write_text(json.dumps(results, indent=2))
print(f"\nSaved {len(results)} results to data/run_results.json")